# `ingestion/crossref.py` — Validation Walkthrough

Validates prerequisite and cross-reference extraction against the FE 501s manual.

Two types are extracted from chunk text:
- **Prerequisites** (`Condition:` lines) — state that must be true before a procedure.
  ~30 occurrences across the manual.  Stored as raw strings; the retrieval layer uses
  them as semantic search queries to surface prerequisite sections.
- **References** (`(see ...)` phrases) — prose pointers to other sections.  Only 3 in
  this manual.  Stored as raw strings rather than resolved section numbers because
  TOC title similarity is too low for reliable automatic resolution.

In [ ]:
import sys
sys.path.insert(0, "..")

import pypdf
from ingestion.parse import parse_manual, parse_toc
from ingestion.index import build_chunks
from ingestion.crossref import annotate_chunks

PDF = "../manuals/FE_501s_2026_US_en_Bundle_RM_069969-000001_05sq_m1du.pdf"

pages  = parse_manual(PDF, extract_images=False)
toc    = parse_toc(pypdf.PdfReader(PDF))
chunks = build_chunks(pages, toc, "FE_501s_2026")
annotate_chunks(chunks, toc)

with_prereqs = [c for c in chunks if c.prerequisites]
with_refs    = [c for c in chunks if c.references]

print(f"Total chunks:              {len(chunks)}")
print(f"Chunks with prerequisites: {len(with_prereqs)}")
print(f"Chunks with references:    {len(with_refs)}")

## 1. Prerequisites

`Condition:` lines describe the state the bike must be in before a procedure begins.
They are the primary dependency signal the agent uses to order repair steps.

In [ ]:
print(f"{'Section':<10} {'Prerequisite'}")
print("-" * 80)
for c in with_prereqs:
    for prereq in c.prerequisites:
        print(f"{c.section:<10} {prereq[:70]}")

## 2. Prerequisite chains

Some sections share the same prerequisite, forming a dependency chain.
For example, several shock absorber sub-procedures all require the shock absorber
to be removed first.  The retrieval layer can use these to pull in upstream sections
automatically when a user asks about a downstream procedure.

In [ ]:
from collections import defaultdict

prereq_to_sections: dict[str, list[str]] = defaultdict(list)
for c in with_prereqs:
    for p in c.prerequisites:
        # Normalise minor variations (capitalisation, trailing punctuation)
        prereq_to_sections[p.lower().rstrip(".,")].append(c.section)

# Show prerequisites shared by more than one section
shared = {k: v for k, v in prereq_to_sections.items() if len(v) > 1}
print(f"Prerequisites shared by multiple sections: {len(shared)}")
print()
for prereq, sections in sorted(shared.items(), key=lambda x: -len(x[1])):
    print(f"{sections}")
    print(f"  → {prereq[:80]}")

## 3. Cross-references (`see ...`)

Only 3 `(see ...)` phrases exist in this manual, all in the engine chapter.  They use
prose titles rather than section numbers, so they are stored as raw strings.
The retrieval layer resolves them via semantic search at query time.

In [ ]:
for c in with_refs:
    print(f"Section {c.section} — {c.section_title}")
    for ref in c.references:
        print(f"  (see) → \"{ref}\"")
    print(f"  Context: {c.text[c.text.find('see'):c.text.find('see')+120].strip()}")
    print()

## 4. Why automatic resolution was not used

The manual uses prose titles in its cross-references, which do not match TOC entries
closely enough for reliable character-level fuzzy matching.  The cell below shows the
top-5 candidate scores for each raw reference — the best match is wrong in both cases.

In [ ]:
import re
from difflib import SequenceMatcher

_STRIP = re.compile(r'\s+(section|chapter)\s*$', re.IGNORECASE)

raw_refs = [
    "Removing the cylinder head and cylinders",
    "chapter on dismantling the alternator side",
]

for raw in raw_refs:
    norm = _STRIP.sub("", raw).lower().strip()
    scores = sorted(
        [(SequenceMatcher(None, norm, e["title"].lower()).ratio(), e["number"], e["title"])
         for e in toc],
        reverse=True,
    )
    print(f'Reference: "{raw}"')
    print(f"  Top matches (none are correct):")
    for score, num, title in scores[:4]:
        print(f"    {score:.3f}  {num:<12} {title}")
    print()

## 5. Metadata schema

Verify that `prerequisites` and `references` appear as expected on the annotated chunks.

In [ ]:
# Spot-check section 18.5.7 — has both a prerequisite and a cross-reference
sample = next(c for c in chunks if c.section == "18.5.7")

print(f"Section      : {sample.section} — {sample.section_title}")
print(f"Prerequisites: {sample.prerequisites}")
print(f"References   : {sample.references}")
print(f"\nText excerpt :\n{sample.text[:400]}")